# 05 — Add a New Stock


## 1 — Bootstrap


In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
SRC = ROOT / 'src'
assert SRC.exists(), f'Could not find src/ at {SRC}'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

import config
from collect import (collect_prices, registry_template, verify_all_raw,
                     verify_raw_file)
from pipeline import TrainConfig, artifacts_exist, predict_latest, train_stock
from validate import run_all_checks

config.ensure_dirs()
pd.set_option('display.width', 140)

print(f'Registered stocks: {config.list_stocks()}')


Registered stocks: ['RELIANCE', 'TCS']


## 2 — Health check


In [2]:
health = run_all_checks(deep=False)


  PROJECT HEALTH CHECK

Dependencies
------------
  [ok] import numpy             core
  [ok] import pandas            core
  [ok] import sklearn           core
  [ok] import joblib            core
  [ok] import ta                indicators
  [ok] import matplotlib        plots
  [!!] import yfinance          missing — downloading new price data unavailable
  [ok] import torch             LSTM / GRU sequence models
  [ok] import streamlit         the dashboard
  [ok] import plotly            the dashboard
  [!!] import transformers      missing — FinBERT sentiment unavailable
  [!!] import vaderSentiment    missing — VADER sentiment unavailable
  [ok] import requests          news collection

Project modules
---------------
  [ok] src/config.py            imports cleanly
  [ok] src/dataio.py            imports cleanly
  [ok] src/features.py          imports cleanly
  [ok] src/dataset.py           imports cleanly
  [ok] src/stationary.py        imports cleanly
  [ok] src/targets.py     

## 3 — Define the stock to add


In [ ]:
NEW_KEY = 'INFY'
NEW_TICKER = 'INFY.NS'
NEW_NAME = 'Infosys'
NEW_START = '2014-01-01'
NEW_END = '2024-01-01'

print(registry_template(NEW_KEY, NEW_TICKER, NEW_NAME,
                        start=NEW_START, end=NEW_END))


## 4 — Paste the block above into `src/config.py`, then restart the kernel


In [ ]:
import importlib

import config
importlib.reload(config)

print(f'Registered stocks: {config.list_stocks()}')
assert NEW_KEY in config.STOCKS, (
    f'{NEW_KEY} is not in config.STOCKS. Paste the block from step 3 into '
    f'src/config.py, save the file, then restart the kernel and re-run.')
print(f'{NEW_KEY} found in registry')


## 5 — Download price history


In [ ]:
path = collect_prices(NEW_KEY, overwrite=False)
print()
result = verify_raw_file(NEW_KEY)


## 6 — Confirm the history is long enough


In [ ]:
if not result.get('ok'):
    raise RuntimeError(f"Download failed: {result.get('problem')}")

if result.get('warning'):
    print('WARNING')
    print(f"  {result['warning']}")
    print()
    print('  The pipeline will still train, at a shorter horizon.')
    print('  For a horizon-3 model, re-run step 5 with an earlier start date.')
else:
    print(f"{NEW_KEY}: {result['rows']} rows over {result['years']} years.")
    print('Sufficient for a horizon-3 model.')


## 7 — Build the dataset


In [ ]:
from dataset import build_dataset
from features import audit_dataset, print_audit
from stationary import add_stationary_features, get_stationary_features

df = build_dataset(NEW_KEY, with_sentiment=False, save=True, verbose=True)
df = add_stationary_features(df).dropna().reset_index(drop=True)
features = get_stationary_features(df)

print_audit(audit_dataset(df, features), title=f'Audit: {NEW_KEY}')


## 8 — Train and save artifacts


In [ ]:
cfg = TrainConfig()   # defaults: logistic, horizon 3, non-overlapping

artifacts = train_stock(NEW_KEY, cfg=cfg, verbose=True)


## 9 — Verify the trained model


In [ ]:
meta = artifacts['metadata']
perm = meta['permutation'] or {}
seeds = meta['seed_robustness'] or {}

summary = pd.Series({
    'stock': meta['stock_key'],
    'rows': meta['n_modelling_rows'],
    'features': meta['n_features'],
    'horizon': meta['config']['horizon'],
    'accuracy': meta['walk_forward_accuracy'],
    'std': meta['walk_forward_std'],
    'baseline': meta['baseline'],
    'edge': meta['edge'],
    'permutation': perm.get('verdict'),
    'seed_verdict': seeds.get('verdict'),
})
print(summary.to_string())
print()

if perm.get('verdict') == 'LEAKAGE':
    print('STOP: permutation test indicates leakage. Do not use this model.')
elif meta['edge'] <= 0:
    print('No demonstrated edge. The model is valid but has no skill on this')
    print('stock. The dashboard will show a warning banner. This is a normal')
    print('and reportable outcome, not a bug.')
else:
    print('Model has a positive, permutation-clean edge.')


## 10 — Offset robustness


In [ ]:
from modeling import evaluate_across_offsets
from targets import build_directional_dataset

horizon = meta['config']['horizon']

if horizon > 1:
    def builder(frame, offset):
        return build_directional_dataset(
            frame.iloc[offset:].reset_index(drop=True),
            horizon=horizon, k=meta['config']['k'], non_overlapping=True)

    offs, edges = evaluate_across_offsets(
        df, features, meta['config']['model_name'],
        dataset_builder=builder, n_offsets=horizon,
        n_splits=meta['config']['n_splits'],
        embargo=meta['config']['embargo'])

    off_df = pd.DataFrame({'offset': offs, 'edge': [round(e, 4) for e in edges]})
    print(off_df.to_string(index=False))
    print()
    if all(e > 0 for e in edges):
        print('Positive at every sampling offset. The edge is robust.')
    else:
        print('Edge flips negative at one or more offsets.')
        print('Report the mean across offsets, not the best one.')
else:
    print('Horizon is 1; there is only one sampling offset to test.')


## 11 — Final check


In [ ]:
print(predict_latest(NEW_KEY))
print()

rows = []
for key in config.list_stocks():
    rows.append({'stock': key, 'trained': artifacts_exist(key)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print('Launch the dashboard with:   streamlit run app.py')


## 12 — Rebuild the comparison report


In [ ]:
import json
from pipeline import load_artifacts

rows = []
for key in config.list_stocks():
    if not artifacts_exist(key):
        continue
    m = load_artifacts(key)['metadata']
    rows.append({
        'stock': key,
        'horizon': m['config']['horizon'],
        'model': m['config']['model_name'],
        'rows': m['n_modelling_rows'],
        'accuracy': m['walk_forward_accuracy'],
        'std': m['walk_forward_std'],
        'baseline': m['baseline'],
        'edge': m['edge'],
        'permutation': (m['permutation'] or {}).get('verdict'),
    })

table = pd.DataFrame(rows).set_index('stock')
table.to_csv(config.REPORTS_DIR / 'all_stocks_summary.csv')
print(table.to_string())
